In [1]:
########## import some useful package ##########

import time
t1 = time.time()

import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from collections import deque

In [2]:
########## read the data ##########

event_path = "/data/mucollider/two_boosted/3TeV/hhvv_500K/delphes_output.root"
data_file = uproot.open(event_path)
tree = data_file["Delphes"]

In [7]:
# 2. Get all available keys from the tree and normalize them to str
# In uproot 3, tree.keys() returns bytes objects (e.g. b'Particle.PID')
tree_keys_raw = tree.keys()
tree_keys_str = [
    k.decode("utf-8") if isinstance(k, bytes) else str(k) for k in tree_keys_raw
]

# 3. Define the desired particle branches
target_branches = [
    "Particle.PID",
    "Particle.Status",
    "Particle.M1",
    "Particle.M2",
    "Particle.D1",
    "Particle.D2",
    "Particle.PT",
    "Particle.Eta",
    "Particle.Phi",
    "Particle.Mass",
]


def match_branch_name(target_name, available_keys):
    """Match a target branch name against available keys in the TTree."""
    for key in available_keys:
        clean_key = key.split(";")[0]  # Strip cycle numbers if present
        if (
            clean_key == target_name
            or clean_key.endswith("/" + target_name)
            or clean_key.endswith("." + target_name)
        ):
            return key
    return target_name


# Create a mapping from standard branch name to actual tree key name
branch_map = {b: match_branch_name(b, tree_keys_str) for b in target_branches}
actual_branches = list(set(branch_map.values()))

# 4. Read data for Event 0 safely across uproot versions
try:
    # Modern uproot syntax (uproot 4 / 5)
    raw_event = tree.arrays(
        actual_branches, entry_start=0, entry_stop=1, library="np"
    )
except TypeError:
    try:
        # Legacy uproot syntax (uproot 3)
        raw_event = tree.arrays(
            actual_branches, entrystart=0, entrystop=1, outputtype=dict
        )
    except TypeError:
        # Fallback using array slicing
        raw_event = tree.arrays(actual_branches)

# 5. Convert keys in returned event dictionary from bytes/varying strings to target standard names
event_0 = {}
for target_name, actual_key in branch_map.items():
    # Check both string key and bytes key in returned dictionary
    if actual_key in raw_event:
        event_0[target_name] = raw_event[actual_key]
    elif actual_key.encode("utf-8") in raw_event:
        event_0[target_name] = raw_event[actual_key.encode("utf-8")]
    elif target_name in raw_event:
        event_0[target_name] = raw_event[target_name]
    elif target_name.encode("utf-8") in raw_event:
        event_0[target_name] = raw_event[target_name.encode("utf-8")]
    else:
        raise KeyError(
            f"Unable to locate branch '{target_name}' in the extracted event data."
        )

# Extract 1D arrays for Event 0
pids = event_0["Particle.PID"][0]
statuses = event_0["Particle.Status"][0]
m1 = event_0["Particle.M1"][0]
m2 = event_0["Particle.M2"][0]
d1 = event_0["Particle.D1"][0]
d2 = event_0["Particle.D2"][0]
pts = event_0["Particle.PT"][0]
etas = event_0["Particle.Eta"][0]
phis = event_0["Particle.Phi"][0]
masses = event_0["Particle.Mass"][0]

# Dictionary for common PDG IDs in Muon Collider HH processes
PDG_NAMES = {
    25: "Higgs(H)",
    11: "e-",
    -11: "e+",
    12: "nu_e",
    -12: "anti_nu_e",
    13: "mu-",
    -13: "mu+",
    14: "nu_mu",
    -14: "anti_nu_mu",
    5: "b",
    -5: "b_bar",
    21: "gluon",
    22: "photon",
    23: "Z0",
    24: "W+",
    -24: "W-",
    1: "d",
    -1: "d_bar",
    2: "u",
    -2: "u_bar",
    3: "s",
    -3: "s_bar",
    4: "c",
    -4: "c_bar",
}


def get_particle_name(pid):
    """Translate PDG ID into a human-readable particle name."""
    return PDG_NAMES.get(pid, f"PID:{pid}")


# 6. Display non-final-state particles (intermediates / hard process)
df_all = pd.DataFrame(
    {
        "Index": range(len(pids)),
        "Name": [get_particle_name(p) for p in pids],
        "Status": statuses,
        "M1": m1,
        "M2": m2,
        "D1": d1,
        "D2": d2,
        "pT [GeV]": pts,
        "Mass [GeV]": masses,
    }
)

print("=== Event 0: Hard Process & Intermediate Particles ===")
# Status == 1 represents final state stable particles; Status != 1 represents intermediate states
print(df_all.to_string(index=False))


# 7. Recursive function to build and print particle decay tree
def print_decay_tree_debug(idx, indent="", visited=None):
    """Trace decay tree while explicitly printing daughter indices (D1, D2) to reveal shared string fragmentation products."""
    if visited is None:
        visited = set()

    if idx < 0 or idx >= len(pids):
        return

    pid = pids[idx]
    name = get_particle_name(pid)
    status = statuses[idx]
    pt = pts[idx]
    mass = masses[idx]
    d_start = d1[idx]
    d_end = d2[idx]

    # Handle already visited particles (common in shared QCD string fragmentation)
    if idx in visited:
        print(f"{indent}[Idx {idx:3d}] {name:<12} ---> (ALREADY VISITED/PRINTED)")
        return

    visited.add(idx)

    # Print current particle info along with its D1/D2 daughter range
    print(
        f"{indent}[Idx {idx:3d}] {name:<12} (Status: {status:2d}, pT: {pt:5.1f} GeV, D1:{d_start:3d}, D2:{d_end:3d})"
    )

    # Recursively trace daughters if daughter range is valid
    if d_start != -1 and d_end != -1:
        for d_idx in range(d_start, d_end + 1):
            if 0 <= d_idx < len(pids):
                print_decay_tree_debug(d_idx, indent + "  │── ", visited)


# 8. Locate all Higgs bosons (PID == 25) and trace their decay chains
print("\n" + "=" * 65)
print("=== Event 0: Higgs Boson Decay Trees ===")
print("=" * 65)

higgs_indices = [i for i, pid in enumerate(pids) if pid == 25]

if not higgs_indices:
    print("No Higgs boson (PID = 25) found in Event 0.")
else:
    for h_idx in higgs_indices:
        # Print parent info if available
        parent_m1 = m1[h_idx]
        parent_name = (
            get_particle_name(pids[parent_m1]) if parent_m1 >= 0 else "None"
        )

        print(
            f"\nTracing decay starting from Higgs at Index {h_idx} (Mother: {parent_name}):"
        )
        print_decay_tree_debug(h_idx)

=== Event 0: Hard Process & Intermediate Particles ===
 Index        Name  Status   M1  M2   D1   D2    pT [GeV]  Mass [GeV]
     0         mu+       4   -1  -1    1    1    0.000000    0.105660
     1         mu+      21    0  -1    4    7    0.000000    0.000000
     2         mu-       4   -1  -1    3    3    0.000000    0.105660
     3         mu-      21    2  -1    4    7    0.000000    0.000000
     4    Higgs(H)      22    1   3    8    9  301.896698  125.000000
     5    Higgs(H)      22    1   3   19   20  223.089127  125.000000
     6       nu_mu       1    1   3   -1   -1  108.286438    0.000000
     7  anti_nu_mu       1    1   3   -1   -1   66.381149    0.000000
     8           b      23    4  -1   12   12  122.375252    4.700000
     9       b_bar      23    4  -1   10   11  181.730087    4.700000
    10       b_bar      51    9  -1   18   18  127.280121    4.700000
    11       gluon      51    9  -1   13   14   56.690960    0.000000
    12           b      52    8  -1

In [14]:
def get_final_higgs_indices(pids, d1_array, d2_array):
    """Find the indices of the final decaying Higgs bosons (right before decaying to b-bbar).

    In Pythia, a Higgs may transition through multiple status codes.
    We pick the Higgs instances whose daughters are NOT Higgs bosons.
    """
    all_higgs = [i for i, pid in enumerate(pids) if pid == 25]
    final_higgs = []

    for h_idx in all_higgs:
        d_start, d_end = d1_array[h_idx], d2_array[h_idx]
        if d_start != -1 and d_end != -1:
            # Check daughter PIDs
            daughter_pids = [
                pids[d] for d in range(d_start, d_end + 1) if 0 <= d < len(pids)
            ]
            # If daughters do not contain Higgs (PID 25), this is a final decaying Higgs
            if 25 not in daughter_pids:
                final_higgs.append(h_idx)

    return final_higgs

def trace_higgs_ancestor_bfs(particle_idx, pids, m1_array, m2_array, target_higgs_map):
    """Trace particle ancestry using Breadth-First Search (BFS) checking BOTH M1 and M2.

    Parameters:
        particle_idx: Index of the final-state particle to trace.
        pids: Array of particle PIDs.
        m1_array: Array of Mother 1 indices.
        m2_array: Array of Mother 2 indices.
        target_higgs_map: Dict mapping Higgs_Index -> Label (e.g., {idx_H1: 1,
          idx_H2: 2})

    Returns:
        Label (1 for Higgs 1, 2 for Higgs 2, -1 for background/neither)
    """
    queue = deque([particle_idx])
    visited = set()

    while queue:
        curr = queue.popleft()

        # Avoid out-of-bounds, invalid indices, or cycles
        if curr < 0 or curr >= len(pids) or curr in visited:
            continue
        visited.add(curr)

        # Check if current ancestor is one of the final Higgs bosons
        if curr in target_higgs_map:
            return target_higgs_map[curr]  # Returns 1 or 2

        # Add BOTH Mother 1 and Mother 2 to the queue to prevent missing paths
        parent1 = m1_array[curr]
        parent2 = m2_array[curr]

        if parent1 >= 0:
            queue.append(parent1)
        if parent2 >= 0 and parent2 != parent1:
            queue.append(parent2)

    return -1  # Particle does not belong to any Higgs (Background)


# ==========================================
# Example Usage on Event 0
# ==========================================
# 1. Identify final decaying Higgs bosons in Event 0
final_higgs_indices = get_final_higgs_indices(pids, d1, d2)
print(f"Detected final decaying Higgs indices in Event 0: {final_higgs_indices}")

# Create a mapping dict: {higgs_index_1: 1, higgs_index_2: 2}
target_higgs_map = {h_idx: label + 1 for label, h_idx in enumerate(final_higgs_indices)}

# 2. Get all final-state particles (Status == 1 in Pythia/Delphes)
final_state_indices = [i for i, status in enumerate(statuses) if status == 1]

# 3. Classify every final-state particle
particle_labels = {}
for p_idx in final_state_indices:
    label = trace_higgs_ancestor_bfs(p_idx, pids, m1, m2, target_higgs_map)
    particle_labels[p_idx] = label

# Print summary
h1_count = sum(1 for l in particle_labels.values() if l == 1)
h2_count = sum(1 for l in particle_labels.values() if l == 2)
bg_count = sum(1 for l in particle_labels.values() if l == -1)

print(f"Classification Result -> Higgs 1 particles: {h1_count}, Higgs 2 particles: {h2_count}, Background: {bg_count}")
print("final state particles label (1: higgs1, 2: higgs2)\n", particle_labels)

Detected final decaying Higgs indices in Event 0: [4, 5]
Classification Result -> Higgs 1 particles: 47, Higgs 2 particles: 54, Background: 2
final state particles label (1: higgs1, 2: higgs2)
 {6: -1, 7: -1, 79: 2, 82: 2, 83: 2, 85: 2, 86: 2, 87: 2, 91: 2, 100: 1, 102: 1, 104: 1, 108: 1, 110: 1, 115: 2, 116: 2, 117: 2, 119: 2, 122: 2, 123: 1, 124: 1, 128: 2, 129: 2, 130: 2, 131: 2, 132: 2, 133: 2, 134: 2, 135: 2, 136: 2, 137: 2, 138: 2, 140: 2, 141: 2, 143: 2, 144: 1, 145: 1, 148: 1, 150: 1, 151: 1, 152: 1, 153: 1, 154: 1, 155: 1, 156: 1, 157: 1, 158: 1, 159: 1, 160: 1, 162: 1, 163: 2, 164: 2, 165: 2, 166: 2, 167: 2, 169: 1, 170: 1, 171: 2, 172: 2, 173: 2, 175: 2, 176: 2, 177: 2, 179: 2, 182: 1, 183: 1, 184: 1, 185: 1, 186: 1, 187: 1, 191: 2, 192: 2, 196: 2, 197: 2, 198: 2, 199: 2, 200: 1, 202: 1, 207: 1, 209: 2, 210: 2, 212: 2, 213: 2, 217: 1, 218: 1, 219: 1, 223: 1, 226: 2, 227: 2, 228: 2, 229: 2, 230: 1, 231: 1, 232: 1, 233: 1, 234: 1, 235: 1, 236: 1, 237: 1, 238: 1, 239: 1, 240: 2

In [5]:
tree.show()

Event                      TStreamerInfo              asdtype('>i4')
Event.fUniqueID            TStreamerBasicType         asjagged(asdtype('>u4'))
Event.fBits                TStreamerBasicType         asjagged(asdtype('>u4'))
Event.Number               TStreamerBasicType         asjagged(asdtype('>i8'))
Event.ReadTime             TStreamerBasicType         asjagged(asdtype('>f4'))
Event.ProcTime             TStreamerBasicType         asjagged(asdtype('>f4'))
Event.ProcessID            TStreamerBasicType         asjagged(asdtype('>i4'))
Event.MPI                  TStreamerBasicType         asjagged(asdtype('>i4'))
Event.Weight               TStreamerBasicType         asjagged(asdtype('>f4'))
Event.CrossSection         TStreamerBasicType         asjagged(asdtype('>f4'))
Event.CrossSectionError    TStreamerBasicType         asjagged(asdtype('>f4'))
Event.Scale                TStreamerBasicType         asjagged(asdtype('>f4'))
Event.AlphaQED             TStreamerBasicType         asjagged